In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os


# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import pandas as pd

# Load the NYC 311 data
nyc311_data_path = "/kaggle/input/question-pairs-dataset/questions.csv"
nyc311_df = pd.read_csv(nyc311_data_path)

# Display the first few rows to understand the data structure
nyc311_df.head()


,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,1,3,4,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,2,5,6,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,3,7,8,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,4,9,10,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0


In [3]:
# Step 1: Remove irrelevant columns
nyc311_df_cleaned = nyc311_df.drop(['id', 'qid1', 'qid2'], axis=1)

# Step 2: Check for missing values
missing_values = nyc311_df_cleaned.isnull().sum()
print("Missing values per column:\n", missing_values)

# Drop rows with missing values (if any)
nyc311_df_cleaned.dropna(inplace=True)

# Step 3: Normalize text (convert to lowercase and remove special characters)
import re

def clean_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # Remove special characters
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces
    return text

# Apply cleaning function to both questions
nyc311_df_cleaned['question1'] = nyc311_df_cleaned['question1'].apply(clean_text)
nyc311_df_cleaned['question2'] = nyc311_df_cleaned['question2'].apply(clean_text)

# Display the cleaned dataset
nyc311_df_cleaned.head()



Missing values per column:
 question1       1
question2       2
is_duplicate    0
dtype: int64


,question1,question2,is_duplicate
0,what is the step by step guide to invest in sh...,what is the step by step guide to invest in sh...,0
1,what is the story of kohinoor kohinoor diamond,what would happen if the indian government sto...,0
2,how can i increase the speed of my internet co...,how can internet speed be increased by hacking...,0
3,why am i mentally very lonely how can i solve it,find the remainder when math2324math is divide...,0
4,which one dissolve in water quikly sugar salt ...,which fish would survive in salt water,0


In [4]:
import pandas as pd

# Load the NYC 311 Customer Service Requests data
nyc311_data_path = "/kaggle/input/nyc-311-customer-service-requests-analysis/NYC311data.csv"
nyc311_df = pd.read_csv(nyc311_data_path)

# Display the first few rows to understand the data structure
nyc311_df.head()


/tmp/ipykernel_23/2567670094.py:5: DtypeWarning: Columns (48,49) have mixed types. Specify dtype option on import or set low_memory=False.
  nyc311_df = pd.read_csv(nyc311_data_path)


,Unique Key,Created Date,Closed Date,Agency,Agency Name,Complaint Type,Descriptor,Location Type,Incident Zip,Incident Address,...,Bridge Highway Name,Bridge Highway Direction,Road Ramp,Bridge Highway Segment,Garage Lot Name,Ferry Direction,Ferry Terminal Name,Latitude,Longitude,Location
0,32310363,12/31/2015 11:59:45 PM,01-01-16 0:55,NYPD,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,Street/Sidewalk,10034.0,71 VERMILYEA AVENUE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.865682,-73.923501,"(40.86568153633767, -73.92350095571744)"
1,32309934,12/31/2015 11:59:44 PM,01-01-16 1:26,NYPD,New York City Police Department,Blocked Driveway,No Access,Street/Sidewalk,11105.0,27-07 23 AVENUE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.775945,-73.915094,"(40.775945312321085, -73.91509393898605)"
2,32309159,12/31/2015 11:59:29 PM,01-01-16 4:51,NYPD,New York City Police Department,Blocked Driveway,No Access,Street/Sidewalk,10458.0,2897 VALENTINE AVENUE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.870325,-73.888525,"(40.870324522111424, -73.88852464418646)"
3,32305098,12/31/2015 11:57:46 PM,01-01-16 7:43,NYPD,New York City Police Department,Illegal Parking,Commercial Overnight Parking,Street/Sidewalk,10461.0,2940 BAISLEY AVENUE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.835994,-73.828379,"(40.83599404683083, -73.82837939584206)"
4,32306529,12/31/2015 11:56:58 PM,01-01-16 3:24,NYPD,New York City Police Department,Illegal Parking,Blocked Sidewalk,Street/Sidewalk,11373.0,87-14 57 ROAD,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.733060,-73.874170,"(40.733059618956815, -73.87416975810375)"


In [5]:
import pandas as pd
import re
import torch
from transformers import pipeline
from datasets import Dataset

# Step 1: Load the NYC 311 Customer Service Requests data
nyc311_data_path = "/kaggle/input/nyc-311-customer-service-requests-analysis/NYC311data.csv"
# Load CSV with low_memory=False to suppress DtypeWarning
nyc311_df = pd.read_csv(nyc311_data_path, low_memory=False)

# Step 2: Remove Irrelevant Columns
columns_to_drop = [
    'Unique Key', 'Closed Date', 'Agency', 'Agency Name', 'Incident Address', 
    'Bridge Highway Name', 'Bridge Highway Direction', 'Road Ramp', 
    'Bridge Highway Segment', 'Garage Lot Name', 'Ferry Direction', 
    'Ferry Terminal Name', 'Location'
]
nyc311_df_cleaned = nyc311_df.drop(columns=columns_to_drop)

# Step 3: Combine and Contextualize Relevant Columns
def create_context(row):
    complaint_type = row['Complaint Type']
    descriptor = row['Descriptor']
    location_type = row['Location Type']
    incident_zip = row['Incident Zip']
    
    context = f"This request is about {complaint_type} with details: {descriptor}, reported at {location_type}, ZIP code {incident_zip}."
    return context

nyc311_df_cleaned['contextualized_complaint'] = nyc311_df_cleaned.apply(create_context, axis=1)

# Step 4: Text Normalization
def clean_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # Remove special characters, keep alphanumeric and spaces
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces
    return text

# Apply cleaning function to the contextualized complaints
nyc311_df_cleaned['contextualized_complaint'] = nyc311_df_cleaned['contextualized_complaint'].apply(clean_text)

# Step 5: Named Entity Recognition (NER) and Replacement using Hugging Face Transformers with GPU support
# Check if GPU is available
device = 0 if torch.cuda.is_available() else -1

# Load NER pipeline from Hugging Face with GPU support
ner_pipeline = pipeline("ner", model="dbmdz/bert-large-cased-finetuned-conll03-english", device=device)

# Convert the cleaned DataFrame to a Hugging Face Dataset
dataset = Dataset.from_pandas(nyc311_df_cleaned[['contextualized_complaint']])

# Apply the NER pipeline in batches to maximize GPU utilization
def apply_ner_batch(examples):
    ner_results = ner_pipeline(examples['contextualized_complaint'])
    for i in range(len(ner_results)):
        for ent in ner_results[i]:
            if ent['entity_group'] in ['LOC', 'ORG', 'PER']:  # Location, Organization, Person
                examples['contextualized_complaint'][i] = examples['contextualized_complaint'][i].replace(ent['word'], '[REDACTED]')
    return examples

# Use the map function to apply NER in batches
dataset = dataset.map(apply_ner_batch, batched=True)

# Convert back to Pandas DataFrame
nyc311_df_cleaned['contextualized_complaint'] = dataset['contextualized_complaint']

# Display the cleaned dataset
nyc311_df_cleaned.head()


config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

Map:   0%|          | 0/300698 [00:00<?, ? examples/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


,Created Date,Complaint Type,Descriptor,Location Type,Incident Zip,Street Name,Cross Street 1,Cross Street 2,Intersection Street 1,Intersection Street 2,...,School State,School Zip,School Not Found,School or Citywide Complaint,Vehicle Type,Taxi Company Borough,Taxi Pick Up Location,Latitude,Longitude,contextualized_complaint
0,12/31/2015 11:59:45 PM,Noise - Street/Sidewalk,Loud Music/Party,Street/Sidewalk,10034.0,VERMILYEA AVENUE,ACADEMY STREET,WEST 204 STREET,NaN,NaN,...,Unspecified,Unspecified,N,NaN,NaN,NaN,NaN,40.865682,-73.923501,this request is about noise streetsidewalk wit...
1,12/31/2015 11:59:44 PM,Blocked Driveway,No Access,Street/Sidewalk,11105.0,23 AVENUE,27 STREET,28 STREET,NaN,NaN,...,Unspecified,Unspecified,N,NaN,NaN,NaN,NaN,40.775945,-73.915094,this request is about blocked driveway with de...
2,12/31/2015 11:59:29 PM,Blocked Driveway,No Access,Street/Sidewalk,10458.0,VALENTINE AVENUE,EAST 198 STREET,EAST 199 STREET,NaN,NaN,...,Unspecified,Unspecified,N,NaN,NaN,NaN,NaN,40.870325,-73.888525,this request is about blocked driveway with de...
3,12/31/2015 11:57:46 PM,Illegal Parking,Commercial Overnight Parking,Street/Sidewalk,10461.0,BAISLEY AVENUE,EDISON AVENUE,B STREET,NaN,NaN,...,Unspecified,Unspecified,N,NaN,NaN,NaN,NaN,40.835994,-73.828379,this request is about illegal parking with det...
4,12/31/2015 11:56:58 PM,Illegal Parking,Blocked Sidewalk,Street/Sidewalk,11373.0,57 ROAD,SEABURY STREET,HOFFMAN DRIVE,NaN,NaN,...,Unspecified,Unspecified,N,NaN,NaN,NaN,NaN,40.733060,-73.874170,this request is about illegal parking with det...


In [6]:
import pandas as pd

# Load the Question-Pairs dataset
questions_data_path = "/kaggle/input/question-pairs-dataset/questions.csv"
questions_df = pd.read_csv(questions_data_path)

# Display the first few rows to understand the data structure
questions_df.head()


,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,1,3,4,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,2,5,6,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,3,7,8,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,4,9,10,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0


In [7]:
# Drop irrelevant columns
questions_df_cleaned = questions_df.drop(columns=['id', 'qid1', 'qid2'])


In [8]:
# Check for missing values in question1 and question2 columns
missing_q1 = questions_df['question1'].isna().sum()
missing_q2 = questions_df['question2'].isna().sum()

print(f"Missing values in question1: {missing_q1}")
print(f"Missing values in question2: {missing_q2}")


Missing values in question1: 1
Missing values in question2: 2


In [9]:
# Fill missing values with an empty string
questions_df['question1'].fillna("", inplace=True)
questions_df['question2'].fillna("", inplace=True)

# Verify there are no more missing values
print(questions_df[['question1', 'question2']].isna().sum())


question1    0
question2    0
dtype: int64


/tmp/ipykernel_23/877690618.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  questions_df['question1'].fillna("", inplace=True)
/tmp/ipykernel_23/877690618.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try usin

In [10]:
import re

def normalize_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # Remove special characters and punctuation
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces
    return text

# Apply normalization to both question1 and question2
questions_df['question1'] = questions_df['question1'].apply(normalize_text)
questions_df['question2'] = questions_df['question2'].apply(normalize_text)

# Display the first few rows to verify normalization
print(questions_df.head())

   id  qid1  qid2                                          question1  \
0   0     1     2  what is the step by step guide to invest in sh...   
1   1     3     4     what is the story of kohinoor kohinoor diamond   
2   2     5     6  how can i increase the speed of my internet co...   
3   3     7     8   why am i mentally very lonely how can i solve it   
4   4     9    10  which one dissolve in water quikly sugar salt ...   

                                           question2  is_duplicate  
0  what is the step by step guide to invest in sh...             0  
1  what would happen if the indian government sto...             0  
2  how can internet speed be increased by hacking...             0  
3  find the remainder when math2324math is divide...             0  
4             which fish would survive in salt water             0  


In [11]:
# Create a new column that contextualizes each pair of questions
def create_context(row):
    question1 = row['question1']
    question2 = row['question2']
    return f"Question 1: {question1} Question 2: {question2}"

# Apply the function to create the contextualized pair column
questions_df['contextualized_pair'] = questions_df.apply(create_context, axis=1)

# Display the first few rows to verify contextualization
print(questions_df[['question1', 'question2', 'contextualized_pair']].head())


                                           question1  \
0  what is the step by step guide to invest in sh...   
1     what is the story of kohinoor kohinoor diamond   
2  how can i increase the speed of my internet co...   
3   why am i mentally very lonely how can i solve it   
4  which one dissolve in water quikly sugar salt ...   

                                           question2  \
0  what is the step by step guide to invest in sh...   
1  what would happen if the indian government sto...   
2  how can internet speed be increased by hacking...   
3  find the remainder when math2324math is divide...   
4             which fish would survive in salt water   

                                 contextualized_pair  
0  Question 1: what is the step by step guide to ...  
1  Question 1: what is the story of kohinoor kohi...  
2  Question 1: how can i increase the speed of my...  
3  Question 1: why am i mentally very lonely how ...  
4  Question 1: which one dissolve in water quikly..

In [12]:
import pandas as pd

# Load the Enron Email dataset
enron_data_path = "/kaggle/input/enron-email-dataset/emails.csv"
enron_df = pd.read_csv(enron_data_path)

# Display the first few rows to understand the data structure
print(enron_df.head())

# Display basic information about the dataset, such as column types and any missing values
print(enron_df.info())


                       file                                            message
0     allen-p/_sent_mail/1.  Message-ID: <18782981.1075855378110.JavaMail.e...
1    allen-p/_sent_mail/10.  Message-ID: <15464986.1075855378456.JavaMail.e...
2   allen-p/_sent_mail/100.  Message-ID: <24216240.1075855687451.JavaMail.e...
3  allen-p/_sent_mail/1000.  Message-ID: <13505866.1075863688222.JavaMail.e...
4  allen-p/_sent_mail/1001.  Message-ID: <30922949.1075863688243.JavaMail.e...
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517401 entries, 0 to 517400
Data columns (total 2 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   file     517401 non-null  object
 1   message  517401 non-null  object
dtypes: object(2)
memory usage: 7.9+ MB
None


In [13]:
import re

# Function to extract the subject and body from the raw email content
def extract_email_content(raw_message):
    # Extract the subject
    subject_match = re.search(r"Subject: (.*)", raw_message)
    subject = subject_match.group(1).strip() if subject_match else ""

    # Extract the body (everything after the first blank line)
    body_split = re.split(r"\n\s*\n", raw_message, maxsplit=1)
    body = body_split[1].strip() if len(body_split) > 1 else ""

    return subject, body

# Apply the extraction function to each row
enron_df[['subject', 'body']] = enron_df['message'].apply(lambda msg: pd.Series(extract_email_content(msg)))

# Drop the original 'message' column as it is no longer needed
enron_df_cleaned = enron_df.drop(columns=['message'])

# Display the first few rows to verify the extraction
print(enron_df_cleaned.head())


                       file    subject  \
0     allen-p/_sent_mail/1.              
1    allen-p/_sent_mail/10.        Re:   
2   allen-p/_sent_mail/100.   Re: test   
3  allen-p/_sent_mail/1000.              
4  allen-p/_sent_mail/1001.  Re: Hello   

                                                body  
0                               Here is our forecast  
1  Traveling to have a business meeting takes the...  
2                     test successful.  way to go!!!  
3  Randy,\n\n Can you send me a schedule of the s...  
4                  Let's shoot for Tuesday at 11:45.  


In [14]:
# Function to normalize text
def normalize_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # Remove special characters and punctuation
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces
    return text

# Apply normalization to both subject and body
enron_df_cleaned['subject'] = enron_df_cleaned['subject'].apply(normalize_text)
enron_df_cleaned['body'] = enron_df_cleaned['body'].apply(normalize_text)

# Display the first few rows to verify normalization
print(enron_df_cleaned.head())


                       file   subject  \
0     allen-p/_sent_mail/1.             
1    allen-p/_sent_mail/10.        re   
2   allen-p/_sent_mail/100.   re test   
3  allen-p/_sent_mail/1000.             
4  allen-p/_sent_mail/1001.  re hello   

                                                body  
0                               here is our forecast  
1  traveling to have a business meeting takes the...  
2                          test successful way to go  
3  randy can you send me a schedule of the salary...  
4                     lets shoot for tuesday at 1145  


In [15]:
# Refined function to create a contextualized version of the email
def refined_create_context(row):
    subject = row['subject']
    body = row['body']
    if subject:
        return f"Subject: {subject}. Body: {body}"
    else:
        return f"Body: {body}"

# Apply the refined function to create the contextualized email column
enron_df_cleaned['contextualized_email'] = enron_df_cleaned.apply(refined_create_context, axis=1)

# Display the first few rows to verify the refined contextualization
print(enron_df_cleaned[['subject', 'body', 'contextualized_email']].head())



    subject                                               body  \
0                                         here is our forecast   
1        re  traveling to have a business meeting takes the...   
2   re test                          test successful way to go   
3            randy can you send me a schedule of the salary...   
4  re hello                     lets shoot for tuesday at 1145   

                                contextualized_email  
0                         Body: here is our forecast  
1  Subject: re. Body: traveling to have a busines...  
2  Subject: re test. Body: test successful way to go  
3  Body: randy can you send me a schedule of the ...  
4  Subject: re hello. Body: lets shoot for tuesda...  


In [16]:
import pandas as pd

# Load the NYC 311 dataset (make sure the path is correct for your environment)
nyc311_data_path = "/kaggle/input/nyc-311-customer-service-requests-analysis/NYC311data.csv"
nyc311_df = pd.read_csv(nyc311_data_path)

# Select the relevant columns and rename them for consistency
nyc311_df_cleaned = nyc311_df[[
    "Created Date", "Complaint Type", "Descriptor", "Location Type", "Incident Zip"
]]
nyc311_df_cleaned.columns = ["created_date", "complaint_type", "descriptor", "location_type", "incident_zip"]

# Display the first few rows to verify the cleaned data
print(nyc311_df_cleaned.head())


/tmp/ipykernel_23/3703299510.py:5: DtypeWarning: Columns (48,49) have mixed types. Specify dtype option on import or set low_memory=False.
  nyc311_df = pd.read_csv(nyc311_data_path)


             created_date           complaint_type  \
0  12/31/2015 11:59:45 PM  Noise - Street/Sidewalk   
1  12/31/2015 11:59:44 PM         Blocked Driveway   
2  12/31/2015 11:59:29 PM         Blocked Driveway   
3  12/31/2015 11:57:46 PM          Illegal Parking   
4  12/31/2015 11:56:58 PM          Illegal Parking   

                     descriptor    location_type  incident_zip  
0              Loud Music/Party  Street/Sidewalk       10034.0  
1                     No Access  Street/Sidewalk       11105.0  
2                     No Access  Street/Sidewalk       10458.0  
3  Commercial Overnight Parking  Street/Sidewalk       10461.0  
4              Blocked Sidewalk  Street/Sidewalk       11373.0  


In [17]:
# Replace NaN or missing values with appropriate defaults
nyc311_df_cleaned = nyc311_df_cleaned.fillna({
    "created_date": "1970-01-01 00:00:00",  # Replace missing dates with a default date
    "complaint_type": "Unknown",            # Replace missing complaint types with 'Unknown'
    "descriptor": "No Descriptor",          # Replace missing descriptors with 'No Descriptor'
    "location_type": "Unknown Location",    # Replace missing location types with 'Unknown Location'
    "incident_zip": "00000"                 # Replace missing ZIP codes with a default
})

# Ensure incident_zip is always a string to prevent issues
nyc311_df_cleaned["incident_zip"] = nyc311_df_cleaned["incident_zip"].astype(str)

# Display first few rows to verify
print(nyc311_df_cleaned.head())


             created_date           complaint_type  \
0  12/31/2015 11:59:45 PM  Noise - Street/Sidewalk   
1  12/31/2015 11:59:44 PM         Blocked Driveway   
2  12/31/2015 11:59:29 PM         Blocked Driveway   
3  12/31/2015 11:57:46 PM          Illegal Parking   
4  12/31/2015 11:56:58 PM          Illegal Parking   

                     descriptor    location_type incident_zip  
0              Loud Music/Party  Street/Sidewalk      10034.0  
1                     No Access  Street/Sidewalk      11105.0  
2                     No Access  Street/Sidewalk      10458.0  
3  Commercial Overnight Parking  Street/Sidewalk      10461.0  
4              Blocked Sidewalk  Street/Sidewalk      11373.0  


In [18]:
nyc311_df_cleaned.to_csv('nyc311_cleaned.csv', index=False)
questions_df.to_csv('question_pairs_cleaned.csv', index=False)
enron_df_cleaned.to_csv('enron_emails_cleaned.csv', index=False)


In [19]:
import os
print(os.listdir('/kaggle/working'))


['nyc311_cleaned.csv', 'enron_emails_cleaned.csv', '__notebook__.ipynb', 'question_pairs_cleaned.csv']


In [20]:
from IPython.display import FileLink

# Display download links for each of the cleaned CSV files
display(FileLink('nyc311_cleaned.csv'))
display(FileLink('question_pairs_cleaned.csv'))
display(FileLink('enron_emails_cleaned.csv'))


/kaggle/working/nyc311_cleaned.csv

/kaggle/working/question_pairs_cleaned.csv

/kaggle/working/enron_emails_cleaned.csv

In [21]:
# Import pandas to load and view the CSV files
import pandas as pd

# Load each cleaned CSV file and display a sample of the content
nyc311_cleaned_df = pd.read_csv('nyc311_cleaned.csv')
questions_cleaned_df = pd.read_csv('question_pairs_cleaned.csv')
enron_emails_cleaned_df = pd.read_csv('enron_emails_cleaned.csv')

# Display the first few rows of each cleaned dataset
print("NYC 311 Cleaned Dataset Sample:")
print(nyc311_cleaned_df.head(), '\n')

print("Question Pairs Cleaned Dataset Sample:")
print(questions_cleaned_df.head(), '\n')

print("Enron Emails Cleaned Dataset Sample:")
print(enron_emails_cleaned_df.head(), '\n')


NYC 311 Cleaned Dataset Sample:
             created_date           complaint_type  \
0  12/31/2015 11:59:45 PM  Noise - Street/Sidewalk   
1  12/31/2015 11:59:44 PM         Blocked Driveway   
2  12/31/2015 11:59:29 PM         Blocked Driveway   
3  12/31/2015 11:57:46 PM          Illegal Parking   
4  12/31/2015 11:56:58 PM          Illegal Parking   

                     descriptor    location_type  incident_zip  
0              Loud Music/Party  Street/Sidewalk       10034.0  
1                     No Access  Street/Sidewalk       11105.0  
2                     No Access  Street/Sidewalk       10458.0  
3  Commercial Overnight Parking  Street/Sidewalk       10461.0  
4              Blocked Sidewalk  Street/Sidewalk       11373.0   

Question Pairs Cleaned Dataset Sample:
   id  qid1  qid2                                          question1  \
0   0     1     2  what is the step by step guide to invest in sh...   
1   1     3     4     what is the story of kohinoor kohinoor diamond

In [22]:
import pandas as pd

# Step 1: Load the cleaned Enron emails dataset
try:
    enron_df_cleaned = pd.read_csv('enron_emails_cleaned.csv')
    print("Dataset loaded successfully.")
except FileNotFoundError:
    print("Error: The file 'enron_emails_cleaned.csv' was not found. Please check the file path.")

# Step 2: Drop unnecessary columns
# Example column 'file' is being dropped; adjust if needed
if 'enron_df_cleaned' in locals():  # Only proceed if the dataset was loaded correctly
    columns_to_drop = ['file']
    enron_df_reduced = enron_df_cleaned.drop(columns=columns_to_drop, errors='ignore')

    # Step 3: Sample a fraction of the dataset (e.g., 10%)
    enron_df_sampled = enron_df_reduced.sample(frac=0.1, random_state=42)

    # Step 4: Save the sampled dataset
    enron_df_sampled.to_csv('enron_emails_sampled.csv', index=False)
    print("Reduced and sampled dataset saved as 'enron_emails_sampled.csv'")


from IPython.display import FileLink

# Display download links for each of the cleaned CSV files

display(FileLink('enron_emails_sampled.csv'))



Dataset loaded successfully.
Reduced and sampled dataset saved as 'enron_emails_sampled.csv'


/kaggle/working/enron_emails_sampled.csv

In [23]:
enron_df_sampled.head()

,subject,body,contextualized_email
427616,re credit derivatives,bill thanks for the info i also spoke with jef...,Subject: re credit derivatives. Body: bill tha...
108773,meter 1591 lamay gaslift,aimee please check meter 1591 lamay gas lift i...,Subject: meter 1591 lamay gaslift. Body: aimee...
355471,re man night again,gcca crawfish and ripoff raffle overpriced pri...,Subject: re man night again. Body: gcca crawfi...
457837,enron 480 1480 charges,keonizip chris per your request here are the 4...,Subject: enron 480 1480 charges. Body: keonizi...
124910,transport deal,im trying to change the receipt meter on deal ...,Subject: transport deal. Body: im trying to ch...
